# Banking Analytics — Data Cleaning & SQLite Load
This notebook ingests the raw LendingClub CSV, cleans and engineers features, then writes the result to a local SQLite database for SQL-based analysis.

## Cell 1 — Imports
Load the three libraries needed for data manipulation, database connectivity, and file-path operations.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import os

## Cell 2 — Load the CSV
Read the raw loan file into a DataFrame; `low_memory=False` prevents pandas from guessing dtypes column-by-column, which avoids mixed-type warnings on large files.

In [ ]:
csv_path = os.path.join('..', 'data', 'loan.csv')

df_raw = pd.read_csv(csv_path, low_memory=False)

print(f'Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
print('\nAll column names:')
for col in df_raw.columns:
    print(f'  {col}')

df_raw.head(3)

## Cell 3 — Filter to Required Columns
Narrow the dataset to the 8 analytically relevant columns and drop rows with null values in the key fields so downstream queries never encounter unexpected NULLs.

In [ ]:
keep_cols = [
    'loan_amnt', 'grade', 'loan_status', 'int_rate',
    'term', 'issue_d', 'annual_inc', 'purpose'
]

df = df_raw[keep_cols].copy()

df = df.dropna(subset=['grade', 'loan_status', 'issue_d'])

print(f'Shape after filtering: {df.shape[0]:,} rows x {df.shape[1]} columns')
print('\nloan_status value counts:')
print(df['loan_status'].value_counts())

## Cell 4 — Parse Dates & Engineer Features
Convert the issue date string to a proper datetime so we can extract year and month for time-series analysis, and create binary default/paid flags that make SQL aggregations and Tableau calculations straightforward.

In [ ]:
# Parse issue_d
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')
df['issue_year']  = df['issue_d'].dt.year.astype(int)
df['issue_month'] = df['issue_d'].dt.month.astype(int)

# Default flag
default_statuses = {
    'Charged Off',
    'Default',
    'Does not meet the credit policy. Status:Charged Off'
}
df['is_default'] = df['loan_status'].isin(default_statuses).astype(int)

# Paid flag
df['is_paid'] = (df['loan_status'] == 'Fully Paid').astype(int)

print('is_default value counts:')
print(df['is_default'].value_counts())
print(f'\nDefault rate: {df["is_default"].mean():.2%}')

print('\nis_paid value counts:')
print(df['is_paid'].value_counts())
print(f'\nPaid rate: {df["is_paid"].mean():.2%}')

print(f'\nFinal shape: {df.shape[0]:,} rows x {df.shape[1]} columns')

## Cell 5 — Write to SQLite
Persist the cleaned DataFrame to a local SQLite database so every subsequent analysis can query it with standard SQL without re-reading the large CSV.

In [ ]:
db_path = os.path.join('..', 'data', 'loans.db')
engine = create_engine(f'sqlite:///{db_path}')

df.to_sql(
    name='loans',
    con=engine,
    if_exists='replace',
    index=False,
    chunksize=10000
)

row_count = df.shape[0]
db_size_mb = os.path.getsize(db_path) / (1024 ** 2)

print(f'Written {row_count:,} rows to loans table in {db_path}')
print(f'loans.db file size: {db_size_mb:.1f} MB')

## Cell 6 — Verify the Write
Read the `loans` table back from SQLite to confirm the row count and column structure survived the write correctly before moving on to analysis.

In [ ]:
df_check = pd.read_sql('SELECT * FROM loans', con=engine)

print(f'Verified shape: {df_check.shape[0]:,} rows x {df_check.shape[1]} columns')
df_check.head(3)